In [3]:
# Run once
%pip install pgmpy kaggle pandas scikit-learn matplotlib seaborn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# run once, goes to ftp.ncdc.noaa.gov/pub/data/gsod/2002 and grabs data
from ftplib import FTP
import os

os.makedirs("../data/raw/gsod", exist_ok=True)
local_tar_2002 = "../data/raw/gsod/gsod_2002.tar"
local_tar_2003 = "../data/raw/gsod/gsod_2003.tar"

for year, local_tar in [(2002, local_tar_2002), (2003, local_tar_2003)]:
    if not os.path.exists(local_tar):
        print(f"Working on {year}...")
        ftp = FTP("ftp.ncdc.noaa.gov")
        ftp.login("ftp", "gavin.burchett@wsu.edu")   
        ftp.cwd(f"/pub/data/gsod/{year}")
        print("Files:", ftp.nlst()[:5])                

        with open(local_tar, "wb") as f:
            ftp.retrbinary(f"RETR gsod_{year}.tar", f.write)
        ftp.quit()
        print(f"Downloaded → {local_tar}")
    else:
        print(f"Already downloaded {year}")

Already downloaded 2002
Working on 2003...


error_perm: 550 /pub/data/gsod/2003: No such file or directory

In [ ]:
# Extract the tar files
import tarfile

extract_dir_2002 = "../data/raw/gsod/2002"
extract_dir_2003 = "../data/raw/gsod/2003"
os.makedirs(extract_dir_2002, exist_ok=True)
os.makedirs(extract_dir_2003, exist_ok=True)

for local_tar, extract_dir in [(local_tar_2002, extract_dir_2002), (local_tar_2003, extract_dir_2003)]:
    with tarfile.open(local_tar, "r") as tar:
        members = tar.getnames()
        tar.extractall(extract_dir)
        print(f"Extracted {len(members)} station files for {local_tar}")
        print("Sample:", members[:3])

C:\Users\gavin\AppData\Local\Temp\ipykernel_15780\1330520785.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


Extracted 8991 station files
Sample: ['.', './010010-99999-2002.op.gz', './478220-99999-2002.op.gz']


In [ ]:
# Read the .op.gz files into a single DataFrame
import pandas as pd
import glob
import gzip

COL_NAMES = [
    "STN", "WBAN", "YEARMODA",
    "TEMP", "TEMP_CNT", "DEWP", "DEWP_CNT",
    "SLP",  "SLP_CNT",  "STP",  "STP_CNT",
    "VISIB","VISIB_CNT","WDSP", "WDSP_CNT",
    "MXSPD","GUST","MAX","MIN","PRCP","SNDP","FRSHTT"
],

# NOAA missing value sentinels per element
SENTINELS = {
    "TEMP": 9999.9, "DEWP": 9999.9, "SLP": 9999.9, "STP": 9999.9,
    "VISIB": 999.9, "WDSP": 999.9,  "MXSPD": 999.9, "GUST": 9999.9,
    "MAX": 9999.9,  "MIN": 9999.9,  "PRCP": 99.99,   "SNDP": 999.9,
}

def read_gsod_file(filepath):
    with gzip.open(filepath, "rt") as f:
        df = pd.read_csv(f, sep=r"\s+", skiprows=1, header=None, names=COL_NAMES)

    # MAX and MIN may carry a trailing '*' flag indicating a provisional value
    df["MAX"] = pd.to_numeric(
        df["MAX"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    df["MIN"] = pd.to_numeric(
        df["MIN"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    # PRCP carries a trailing letter flag (A–I indicating accumulation window)
    df["PRCP"] = pd.to_numeric(
        df["PRCP"].astype(str).str.replace(r"[A-Za-z]", "", regex=True), errors="coerce"
    )

    # Apply missing sentinels
    for col, sentinel in SENTINELS.items():
        df[col] = df[col].replace(sentinel, pd.NA)

    return df

for year, extract_dir in [(2002, extract_dir_2002), (2003, extract_dir_2003)]:
    gz_files = glob.glob(f"{extract_dir}/*.op.gz")
    print(f"Found {len(gz_files)} station files for {year}")

    df_raw = pd.concat([read_gsod_file(f) for f in gz_files], ignore_index=True)
    print(f"Total records for {year}: {len(df_raw):,}")
    df_raw.to_csv(f"../data/processed/gsod_{year}.csv", index=False)
    print(f"Saved → ../data/processed/gsod_{year}.csv")

Found 8990 station files
Total records: 2,788,413


,STN,WBAN,YEARMODA,TEMP,TEMP_CNT,DEWP,DEWP_CNT,SLP,SLP_CNT,STP,...,VISIB_CNT,WDSP,WDSP_CNT,MXSPD,GUST,MAX,MIN,PRCP,SNDP,FRSHTT
0,10010,99999,20020101,25.8,7,19.1,7,1005.8,7,1004.6,...,5,15.0,7,23.3,999.9,28.2,24.6,0.03,NaN,1000
1,10010,99999,20020102,36.5,8,33.2,8,998.5,8,997.4,...,7,15.5,8,27.2,999.9,44.8,25.7,0.2,NaN,11000
2,10010,99999,20020103,32.3,7,29.5,7,1001.2,7,1000.0,...,5,10.8,7,27.2,999.9,36.9,29.3,0.07,NaN,110000
3,10010,99999,20020104,35.8,8,30.5,8,1011.2,8,1010.0,...,6,13.4,8,23.3,999.9,41.0,29.3,0.03,NaN,101000
4,10010,99999,20020105,34.4,8,33.0,8,1003.1,8,1002.0,...,6,7.3,8,11.7,999.9,36.0,32.5,0.24,NaN,110000


In [ ]:
# Parse YEARMODA → proper date
df_raw["DATE"] = pd.to_datetime(df_raw["YEARMODA"].astype(str), format="%Y%m%d")

# FRSHTT indicates the following Fog | Rain | Snow | Hail | Thunder | Tornado
df_raw["FRSHTT"]  = df_raw["FRSHTT"].astype(str).str.zfill(6)
df_raw["FOG"]     = df_raw["FRSHTT"].str[0].astype(int)
df_raw["RAIN"]    = df_raw["FRSHTT"].str[1].astype(int)
df_raw["SNOW"]    = df_raw["FRSHTT"].str[2].astype(int)
df_raw["HAIL"]    = df_raw["FRSHTT"].str[3].astype(int)
df_raw["THUNDER"] = df_raw["FRSHTT"].str[4].astype(int)
df_raw["TORNADO"] = df_raw["FRSHTT"].str[5].astype(int)

print(df_raw.dtypes)
print(df_raw[["DATE","TEMP","DEWP","SLP","VISIB","WDSP","MXSPD","GUST",
               "MAX","MIN","PRCP","SNDP","RAIN","SNOW","FOG"]].describe())

STN                   int64
WBAN                  int64
YEARMODA              int64
TEMP                float64
TEMP_CNT              int64
DEWP                 object
DEWP_CNT              int64
SLP                  object
SLP_CNT               int64
STP                  object
STP_CNT               int64
VISIB                object
VISIB_CNT             int64
WDSP                 object
WDSP_CNT              int64
MXSPD                object
GUST                float64
MAX                  object
MIN                  object
PRCP                 object
SNDP                 object
FRSHTT               object
DATE         datetime64[ns]
FOG                   int64
RAIN                  int64
SNOW                  int64
HAIL                  int64
THUNDER               int64
TORNADO               int64
dtype: object
                                DATE          TEMP          GUST  \
count                        2788413  2.788413e+06  2.788413e+06   
mean   2002-07-01 18:56:34.733563392  

In [ ]:
# Save the cleaned DataFrame to CSV
# Need to check for handling missing vals and some more cleaning, some format changes maybe
# Need to check more on what Bayes networks want for data format and input types
# have not checked outliers yet either
# need station filtering maybe? depends on the route we want to go.

os.makedirs("../data/processed", exist_ok=True)
df_raw.to_csv("../data/processed/gsod_2003.csv", index=False)
print("Saved → ../data/processed/gsod_2003.csv")

Saved → ../data/processed/gsod_2002.csv


**Notes from meeting

need a network structure

- pressure and temperature are more "fundamental" factors
- precipitation will be rain and snow, temp and pressure related
- wind will be pressure and temperatre impacted, not by precip
- dewpoint - idk, need to learn more weather
- fog plays in ?????
- for Bayes there should be binning for each type but need classifiers?
- pressure = high/low
- temp = cold, avg, warm
- wind = none, low, high ???
- dew point ??
- fog, rain, snow can either be bools or maybe light/heavy/none classifications?    For the precipitations have precip be light/heavy then can just be bools for the type of precip? fog is still ??
- 

Meeting Notes:
double check for provided bayesian graph information, otherwise look into other sources for graph templates. 
If we can't find the information ask the TA for assistance

take a look at a simplified dynamic bayesian network through another model. using t-1 to see hsitory, this may simplify our process.
^ The hidden markov model is what may be good here.